# Étape 1 – Géocodage des 35 destinations

**Objectif** : obtenir pour chaque destination de l'énoncé un **identifiant unique** et ses **coordonnées GPS**
(latitude, longitude), indispensables pour interroger l'API météo (étape 2) et cartographier les résultats (étape 6).

**Source** : API [Nominatim](https://nominatim.org/release-docs/develop/api/Search/) (OpenStreetMap), gratuite et sans clé.

**Compétence du référentiel** : C3 – collecter des données provenant de différentes sources (ici : une API web).

**Sortie** : `data/raw/cities.csv`

| Règle d'usage de Nominatim | Mise en œuvre dans ce notebook |
|---|---|
| 1 requête par seconde maximum | pause de 1,1 s entre deux appels |
| User-Agent identifiant l'application | en-tête `User-Agent` dédié au projet |
| Pas d'usage intensif | 35 requêtes, une seule exécution suffit |

## 1. Imports et chemins

In [1]:
import time
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient requirements.txt (racine du projet)."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt absent)")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = RAW_DIR / "cities.csv"

print("Racine du projet :", PROJECT_ROOT)
print("Fichier de sortie :", OUTPUT_FILE)

Racine du projet : c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip
Fichier de sortie : c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip\data\raw\cities.csv


## 2. Liste des destinations (périmètre de l'énoncé)

La liste est reprise **telle quelle** depuis l'énoncé. L'ordre de la liste sert à attribuer l'identifiant `city_id`
(1 à 35) : il est ainsi **stable et reproductible**, ce qui est important car cet identifiant servira de **clé**
pour relier la météo et les hôtels à chaque ville (et de clé primaire dans la base RDS).

In [2]:
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris", "Amiens", "Lille",
    "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar", "Eguisheim", "Besancon", "Dijon",
    "Annecy", "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille",
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes", "Saintes Maries de la mer",
    "Collioure", "Carcassonne", "Ariege", "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle",
]

print(f"{len(CITIES)} destinations")
assert len(CITIES) == len(set(CITIES)) == 35, "La liste doit contenir 35 destinations uniques"

35 destinations


## 3. Fonction d'appel à Nominatim

Choix des paramètres :
- `countrycodes=fr` : limite la recherche à la France (évite par exemple un « Paris » au Texas) ;
- `limit=1` : on garde le résultat jugé le plus pertinent par Nominatim ;
- `format=jsonv2` : renvoie aussi `category`, `type` et `addresstype`, qui permettent de **contrôler la nature**
  du lieu trouvé (ville, département, site touristique…).

La fonction ne lève pas d'erreur bloquante : en cas d'échec, elle renvoie `None` et le problème est signalé.

In [3]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
HEADERS = {"User-Agent": "kayak-plan-your-trip/1.0 (projet pedagogique Jedha - certification RNCP35288)"}
PAUSE_SECONDS = 1.1  # politique d'usage : 1 requête/seconde maximum


def geocode(place: str) -> dict | None:
    """Renvoie le premier résultat Nominatim pour `place` (en France), ou None en cas d'échec."""
    params = {"q": place, "format": "jsonv2", "countrycodes": "fr", "limit": 1}
    try:
        response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
        response.raise_for_status()  # lève une erreur si le code HTTP n'est pas 2xx
    except requests.RequestException as error:
        print(f"  ⚠️ Erreur réseau / HTTP pour '{place}' : {error}")
        return None

    results = response.json()
    if not results:
        print(f"  ⚠️ Aucun résultat pour '{place}'")
        return None
    return results[0]


# Test sur une seule ville avant de lancer la boucle complète
test = geocode("Paris")
test

{'place_id': 98427244,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'relation',
 'osm_id': 7444,
 'lat': '48.8588897',
 'lon': '2.3200410',
 'category': 'boundary',
 'type': 'administrative',
 'place_rank': 15,
 'importance': 0.897098092136026,
 'addresstype': 'suburb',
 'name': 'Paris',
 'display_name': 'Paris, Île-de-France, France métropolitaine, France',
 'boundingbox': ['48.8155755', '48.9021560', '2.2241220', '2.4697602']}

## 4. Géocodage des 35 destinations

Environ 40 secondes d'exécution (35 appels espacés de 1,1 s).
On conserve, en plus des coordonnées, le nom complet trouvé (`display_name`) et la nature du lieu,
pour pouvoir **vérifier** chaque résultat à l'étape suivante.

In [4]:
records = []

for city_id, city_name in enumerate(CITIES, start=1):
    result = geocode(city_name)
    records.append({
        "city_id": city_id,
        "city_name": city_name,
        "latitude": float(result["lat"]) if result else None,
        "longitude": float(result["lon"]) if result else None,
        "display_name": result.get("display_name") if result else None,
        "category": result.get("category") if result else None,
        "place_type": result.get("type") if result else None,
        "address_type": result.get("addresstype") if result else None,
    })
    print(f"{city_id:>2}. {city_name:<30} -> {'OK' if result else 'ÉCHEC'}")
    time.sleep(PAUSE_SECONDS)

cities = pd.DataFrame(records)
cities.head()

 1. Mont Saint Michel              -> OK
 2. St Malo                        -> OK
 3. Bayeux                         -> OK
 4. Le Havre                       -> OK
 5. Rouen                          -> OK
 6. Paris                          -> OK
 7. Amiens                         -> OK
 8. Lille                          -> OK
 9. Strasbourg                     -> OK
10. Chateau du Haut Koenigsbourg   -> OK
11. Colmar                         -> OK
12. Eguisheim                      -> OK
13. Besancon                       -> OK
14. Dijon                          -> OK
15. Annecy                         -> OK
16. Grenoble                       -> OK
17. Lyon                           -> OK
18. Gorges du Verdon               -> OK
19. Bormes les Mimosas             -> OK
20. Cassis                         -> OK
21. Marseille                      -> OK
22. Aix en Provence                -> OK
23. Avignon                        -> OK
24. Uzes                           -> OK
25. Nimes       

,city_id,city_name,latitude,longitude,display_name,category,place_type,address_type
0,1,Mont Saint Michel,48.635954,-1.511460,"Mont Saint-Michel, Le Mont-Saint-Michel, Avran...",place,islet,islet
1,2,St Malo,48.649518,-2.026041,"Saint-Malo, Ille-et-Vilaine, Bretagne, France ...",boundary,administrative,town
2,3,Bayeux,49.276462,-0.702474,"Bayeux, Calvados, Normandie, France métropolit...",boundary,administrative,town
3,4,Le Havre,49.493898,0.107973,"Le Havre, Seine-Maritime, Normandie, France mé...",boundary,administrative,city
4,5,Rouen,49.440459,1.093966,"Rouen, Seine-Maritime, Normandie, France métro...",boundary,administrative,city


## 5. Contrôle qualité

Trois vérifications avant d'enregistrer :
1. **Complétude** : aucune coordonnée manquante ;
2. **Cohérence géographique** : toutes les coordonnées sont en France métropolitaine (cadre approximatif) ;
3. **Nature du lieu** : lecture du type trouvé. Certaines « destinations » de l'énoncé ne sont pas des villes
   (« Ariege » est un département, « Gorges du Verdon » un site naturel, « Chateau du Haut Koenigsbourg » un monument) :
   il faut s'assurer que le point trouvé est pertinent pour la météo et la recherche d'hôtels.

In [7]:
# 1. Complétude
missing = cities[cities[["latitude", "longitude"]].isna().any(axis=1)]
print(f"Coordonnées manquantes : {len(missing)}")
display(missing)

# 2. Cohérence géographique (cadre approximatif de la France métropolitaine)
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 41.0, 51.5, -5.5, 10.0
located = cities.dropna(subset=["latitude", "longitude"])
out_of_france = located[
    ~located["latitude"].between(LAT_MIN, LAT_MAX) | ~located["longitude"].between(LON_MIN, LON_MAX)
]
print(f"Hors France métropolitaine : {len(out_of_france)}")
display(out_of_france)

Coordonnées manquantes : 0


,city_id,city_name,latitude,longitude,display_name,category,place_type,address_type


Hors France métropolitaine : 0


,city_id,city_name,latitude,longitude,display_name,category,place_type,address_type


In [8]:
# 3. Nature des lieux trouvés : à relire ligne par ligne
pd.set_option("display.max_colwidth", 90)
cities[["city_id", "city_name", "address_type", "category", "place_type", "display_name"]]

,city_id,city_name,address_type,category,place_type,display_name
0,1,Mont Saint Michel,islet,place,islet,"Mont Saint-Michel, Le Mont-Saint-Michel, Avranches, Manche, Normandie, France métropol..."
1,2,St Malo,town,boundary,administrative,"Saint-Malo, Ille-et-Vilaine, Bretagne, France métropolitaine, 35400, France"
2,3,Bayeux,town,boundary,administrative,"Bayeux, Calvados, Normandie, France métropolitaine, 14400, France"
3,4,Le Havre,city,boundary,administrative,"Le Havre, Seine-Maritime, Normandie, France métropolitaine, France"
4,5,Rouen,city,boundary,administrative,"Rouen, Seine-Maritime, Normandie, France métropolitaine, France"
5,6,Paris,suburb,boundary,administrative,"Paris, Île-de-France, France métropolitaine, France"
6,7,Amiens,city,boundary,administrative,"Amiens, Somme, Hauts-de-France, France métropolitaine, 80000, France"
7,8,Lille,city,boundary,administrative,"Lille, Nord, Hauts-de-France, France métropolitaine, France"
8,9,Strasbourg,city,boundary,administrative,"Strasbourg, Bas-Rhin, Collectivité européenne d'Alsace, Grand Est, France métropolitai..."
9,10,Chateau du Haut Koenigsbourg,historic,historic,castle,"Château du Haut-Kœnigsbourg, Rampe PMR, Schaflager, Orschwiller, Sélestat-Erstein, Bas..."


### Analyse des résultats
- 35 destinations géocodées, aucune valeur manquante, toutes situées en France métropolitaine.
- 29 destinations sont des communes (`boundary / administrative`).
- Cas particuliers conservés, car fidèles au périmètre de l'énoncé :
  - **Mont Saint Michel** (îlot), **Chateau du Haut Koenigsbourg** (château) et **Gorges du Verdon** (gorge) sont des sites touristiques et non des villes ;
  - **Ariege** correspond au département (point = centre géographique) ;
  - **Paris** est classé `suburb` par OpenStreetMap, mais il s'agit bien de la commune ;
  - **Biarritz** est rattaché à une limite « politique » (canton), dont le point reste situé à Biarritz.
- Limite : pour les destinations qui ne sont pas des villes, la météo est celle d'un point représentatif, et la recherche d'hôtels (étape 3) portera sur leurs environs.

## 6. Vérification visuelle sur une carte

Un contrôle visuel complète les tests : un point mal placé (mer, mauvaise région) saute aux yeux.

In [9]:
fig = px.scatter_map(
    cities.dropna(subset=["latitude", "longitude"]),
    lat="latitude",
    lon="longitude",
    hover_name="city_name",
    hover_data={"city_id": True, "address_type": True, "latitude": ":.4f", "longitude": ":.4f"},
    zoom=4.3,
    height=650,
    map_style="open-street-map",
    title="Étape 1 – Localisation des 35 destinations (contrôle)",
)
fig.show()

## 7. Enregistrement

Le fichier est enregistré dans `data/raw/` : il s'agit d'une **donnée brute** issue de la collecte.
Il sera envoyé dans le Data Lake S3 (zone `raw/`) à l'étape 4.

In [10]:
cities.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"{len(cities)} lignes enregistrées dans {OUTPUT_FILE}")
pd.read_csv(OUTPUT_FILE).head()

35 lignes enregistrées dans c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip\data\raw\cities.csv


,city_id,city_name,latitude,longitude,display_name,category,place_type,address_type
0,1,Mont Saint Michel,48.635954,-1.511460,"Mont Saint-Michel, Le Mont-Saint-Michel, Avranches, Manche, Normandie, France métropol...",place,islet,islet
1,2,St Malo,48.649518,-2.026041,"Saint-Malo, Ille-et-Vilaine, Bretagne, France métropolitaine, 35400, France",boundary,administrative,town
2,3,Bayeux,49.276462,-0.702474,"Bayeux, Calvados, Normandie, France métropolitaine, 14400, France",boundary,administrative,town
3,4,Le Havre,49.493898,0.107973,"Le Havre, Seine-Maritime, Normandie, France métropolitaine, France",boundary,administrative,city
4,5,Rouen,49.440459,1.093966,"Rouen, Seine-Maritime, Normandie, France métropolitaine, France",boundary,administrative,city
